# Lab 6, part 2: Decision trees

Part 1 separated two classes by cutting the plane with a single straight line.
A decision tree asks one question about one feature at a
time, and each answer narrows the group of passengers under consideration until
what remains is mostly one class.

The quantity that decides which question to ask is **entropy**, which you met in
Lab 5 as a measure of how spread out a belief was. Entropy is a fundamental quantity in physics and information theory, and is used widely in machine learning as well.
In Lab 5 it measured ignorance about where the robot was. Here it measures how
mixed a set of labels is.

## Setup

This notebook needs scikit-learn. If the next
cell raises `ModuleNotFoundError`, quit JupyterLab and run:

```
conda activate csc371
conda install -c conda-forge scikit-learn
```

Then start JupyterLab again from this folder. Note that the package installs
under the name `scikit-learn` and imports under the name `sklearn`. That
mismatch is normal and it catches everyone once.

In [ ]:
# PROVIDED
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

print(f"scikit-learn {sklearn.__version__}")

## Part 0: Eight passengers

Consider 8 passengers. Each observational unit has a sex, a passenger class,
and whether they survived.

| # | Sex | Pclass | Survived |
|---|--------|--------|----------|
| 1 | female | 1 | 1 |
| 2 | female | 1 | 1 |
| 3 | female | 3 | 1 |
| 4 | male | 1 | 1 |
| 5 | male | 1 | 0 |
| 6 | male | 3 | 0 |
| 7 | male | 3 | 0 |
| 8 | male | 3 | 0 |

Four survived and four did not. A tree has to choose which question to ask
first: "is this passenger female?" or "was this passenger in first class?" Is one question measurably better than the other?

In [ ]:
# PROVIDED
eight = pd.DataFrame({
    "Sex":      ["female", "female", "female", "male", "male", "male", "male", "male"],
    "Pclass":   [1, 1, 3, 1, 1, 3, 3, 3],
    "Survived": [1, 1, 1, 1, 0, 0, 0, 0],
})
print(eight.to_string(index=False))
print(f"\n{eight.Survived.sum()} of {len(eight)} survived")

## Part 1: The entropy of a set of labels

Entropy measures how mixed a collection is, in bits. For a set of labels where
each distinct label $c$ makes up a proportion $p_c$ of the set,

$$H = -\sum_c p_c \log_2 p_c$$

There are three special cases which are instructive:

* All labels the same: $H = 0$ bits. There is nothing left to learn.
* Half and half: $H = 1$ bit. Maximally mixed, for two classes.
* Three quarters or one quarter: $H \approx 0.81$ bits. 

This is the same function from `lab5kit/measures.py`. In Lab 5 it
took a belief, which was a dictionary of probabilities that already summed to 1.
Here it takes the raw labels, so you have to compute the proportions first.

### Exercise 1

Write `entropy(labels)`, which takes a sequence of labels and returns its entropy
in bits.

A label with proportion 0 contributes nothing to the sum, but $\log_2 0$ is
undefined, so skip any label whose proportion is 0 rather than letting Python
raise and error

In [ ]:
# Your code here.

In [ ]:
# PROVIDED: three cases you can check by eye
try:
    print(f"all eight passengers    {entropy(eight.Survived):.4f} bits   (expect 1.0)")
    print(f"the three women         {entropy([1, 1, 1]):.4f} bits   (expect 0.0)")
    print(f"three of four survived  {entropy([1, 1, 1, 0]):.4f} bits   (expect 0.8113)")
except NameError:
    print("entropy is not defined yet. Write it in the cell above.")

<details><summary>Hint 1: the shape</summary>

One loop over the distinct labels. For each one, compute what proportion of the
set it makes up, then subtract `proportion * math.log2(proportion)` from a
running total. `np.unique(labels)` gives you the distinct values, and
`(labels == value).mean()` gives you the proportion in one step.

</details>

<details><summary>Hint 2: the answers</summary>

`entropy(eight.Survived)` is exactly **1.0**, because four of eight survived.
`entropy([1, 1, 1])` is **0.0**. `entropy([1, 1, 1, 0])` is **0.8113**.

</details>

## Part 2: One split, by hand

Splitting on a feature divides the eight passengers into groups, one per value of
that feature. Each group has its own entropy. To score the split as a whole, take
the **weighted average** of the group entropies, weighting each group by its share
of the rows:

$$H_{\text{after}} = \sum_{\text{groups}} \frac{n_{\text{group}}}{n_{\text{total}}} H(\text{group})$$

The **information gain** is how much entropy the split removed:

$$\text{gain} = H_{\text{before}} - H_{\text{after}}$$

A tree evaluates every available split, computes this number for each, and takes
the largest. Then it repeats the process recursively.

### Exercise 2 (provided)

Write `information_gain(data, feature)`, which returns the information gain in
bits from splitting `data` on `feature`. Assume the label is in a column named
`Survived`.

Then print the gain for splitting the eight passengers on `Sex` and on `Pclass`,
and say which question the tree should ask first.

In [ ]:
# PROVIDED
def information_gain(data, feature):
    """The entropy removed by splitting data on feature, in bits."""
    before = entropy(data.Survived)
    after = 0.0
    for value in data[feature].unique():
        group = data[data[feature] == value]
        after += len(group) / len(data) * entropy(group.Survived)
    return before - after


for feature in ["Sex", "Pclass"]:
    print(f"splitting on {feature:7} gains {information_gain(eight, feature):.4f} bits")

Your result should suggest that it is best to split by sex at the first step of the decision tree. It doesn't necessarily mean that splitting by class isn't useful--just that it isn't as useful on the first step.

## Part 3: The same split, from scikit-learn

We will use `DecisionTreeClassifier` with two details that will make it equivalent to what you just did:

* `criterion="entropy"` tells it to score splits with the formula from Part 1.
  Its default is `"gini"`, which is a different measure of mixing that usually
  picks the same splits but reports different numbers.
* `max_depth=1` stops it after a single split, which is all you computed by hand.

It also requires numbers rather than strings, so `Sex` becomes an `is_male`
column that is 1 for male and 0 for female.

In [ ]:
# PROVIDED
features = pd.DataFrame({
    "is_male": (eight.Sex == "male").astype(int),
    "Pclass": eight.Pclass,
})

stump = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=0)
stump.fit(features, eight.Survived)

chosen = features.columns[stump.tree_.feature[0]]
print(f"scikit-learn split on : {chosen}")
print(f"at threshold          : {stump.tree_.threshold[0]:.2f}")
print(f"root entropy          : {stump.tree_.impurity[0]:.4f} bits")

figure, axis = plt.subplots(figsize=(7, 4))
plot_tree(stump, feature_names=features.columns, class_names=["died", "survived"],
          filled=True, impurity=True, ax=axis)
plt.show()

It chose `is_male`, and it reports the root entropy as 1.0 bits, which is the
number you computed in Exercise 1.

The threshold of 0.50 deserves a look. A tree only ever asks whether a feature is
less than or equal to some number, so to ask "is this passenger female?" about a
0/1 column it asks `is_male <= 0.50`. Every female passenger answers yes and
every male passenger answers no, so the question is the same one you asked. The
odd-looking threshold is an artifact of encoding a category as a number.

In the figure, `samples` is how many passengers reached that node, `value` is how
many of them died and survived in that order, and `entropy` is the mixing you
computed. The left child has entropy 0.0 and holds the three women. It is the
leaf from Part 2.

## Part 4: A real tree

Eight rows and two features was a hand calculation. Now use all 891 passengers
and six features.

One of those features needs a decision made about it. `Age` is missing for 177 of
the 891 passengers, and a tree cannot split on a value that is not there. The
cell below fills those 177 with the median age of everyone else, which is 28.
That is a choice, not a fact about the data, and it is worth remembering that it
was made: 20% of the age column is now an invented number that is identical in
every one of those rows.

In [ ]:
# PROVIDED
titanic = pd.read_csv("../lab4-probability/titanic.csv") # fix the path if this doesn't work

passengers = pd.DataFrame({
    "is_male": (titanic.Sex == "male").astype(int),
    "Pclass": titanic.Pclass,
    "Age": titanic.Age.fillna(titanic.Age.median()),
    "Fare": titanic.Fare,
    "SibSp": titanic.SibSp,
    "Parch": titanic.Parch,
})
survived = titanic.Survived

print(f"{len(passengers)} passengers, {passengers.shape[1]} features")
print(f"ages filled in: {titanic.Age.isna().sum()} at the median of "
      f"{titanic.Age.median():.0f}")
print()
print(passengers.head().to_string())

### Holding data back

As in Lab 4, the model must not be judged on the rows it learned from. The split
below holds back 20% of the passengers, and that 20% is not looked at until the
model is finished.

`stratify=survived` keeps the proportion of survivors the same in both halves, so
the test set is not accidentally easier or harder than the training set.
`random_state=0` fixes the shuffle, so your numbers will match the ones quoted in
the hints.

In [ ]:
# PROVIDED
train_X, test_X, train_y, test_y = train_test_split(
    passengers, survived, test_size=0.2, random_state=0, stratify=survived)

print(f"train: {len(train_X)} passengers")
print(f"test:  {len(test_X)} passengers")
print(f"\nalways guessing 'died' is right {(test_y == 0).mean():.1%} "
      f"of the time on the test set")

That last number is the baseline, and it is the same idea as the spam classifier
that always guessed "ham" in Lab 4. A tree that scores below 61.5% has learned
something worse than nothing.

### Exercise 3 (provided)

Fit a `DecisionTreeClassifier` with `criterion="entropy"`, `max_depth=3` and
`random_state=0` to the training passengers. Print its accuracy on the training
set and on the test set.

`model.score(X, y)` returns accuracy directly, so you do not need to count
anything by hand.

In [ ]:
# PROVIDED
tree = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=0)
tree.fit(train_X, train_y)

print(f"train accuracy {tree.score(train_X, train_y):.4f}")
print(f"test accuracy  {tree.score(test_X, test_y):.4f}")

In [ ]:
# PROVIDED: the tree itself
try:
    figure, axis = plt.subplots(figsize=(20, 9))
    plot_tree(tree, feature_names=passengers.columns,
              class_names=["died", "survived"], filled=True,
              impurity=False, fontsize=9, ax=axis)
    plt.show()

    print("root split:", passengers.columns[tree.tree_.feature[0]],
          "<=", round(float(tree.tree_.threshold[0]), 2))
    print(f"leaves: {tree.get_n_leaves()}")
except NameError:
    print("tree is not defined yet. Fit it in the cell above.")

### Exercise 4

The decision tree results in an interpretable tree which is unusual for a machine learning model.

Answer both of these in the markdown cell below.

Pick any one path from the root to a leaf and write it as an
English sentence, in the form "a passenger who was ..., and ..., was predicted
to ...". Include how many training passengers followed that path.

*Your answer here.*

## Part 5: How deep should the tree be?

`max_depth=3` was handed to you with no justification. Depth is the main control
on how complicated a tree is allowed to get: depth 1 asks a single question,
while an unlimited depth lets the tree keep splitting until every leaf is pure.

### Predict first

Before running anything, commit to an answer in the cell below.

1. As `max_depth` rises from 1 to unlimited, what does **training** accuracy do?
2. What does **test** accuracy do?
3. Do the two mirror each other, or do they differ qualitatively?

*Your prediction here.*

### Exercise 5

For each depth in `depths` below, fit a tree on the training passengers and
record four things: the depth, its training accuracy, its test accuracy, and its
number of leaves, which `model.get_n_leaves()` returns. Collect them in a
`DataFrame` with columns `depth`, `train`, `test` and `leaves`, and print it.

Keep `criterion="entropy"` and `random_state=0` throughout, so depth is the only
thing changing. `max_depth=None` means no limit.

Store the depth as `str(depth)` rather than `depth`. A column holding both whole
numbers and `None` is not numeric, so pandas would convert it to floats and print
the last row's depth as `NaN`.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]

# Your code here.

<details><summary>Hint 1: the shape</summary>

One loop over `depths`. Inside it, build a classifier with that `max_depth`, fit
it, and append a dictionary of the four values to a list. After the loop,
`pd.DataFrame(rows)` turns the list of dictionaries into a table. Name it
`sweep`, because the next provided cell expects that name.

</details>

<details><summary>Hint 2: the answers</summary>

Training accuracy climbs from 0.7893 at depth 1 to **0.9831** with no limit. Test
accuracy climbs from 0.7765 to a peak of **0.8156 at depth 4**, then falls back
to **0.7263** with no limit, which is worse than the depth-1 tree managed.

</details>

In [ ]:
# PROVIDED: the picture of those two columns
try:
    labels = [str(d) for d in sweep.depth]
    positions = range(len(labels))

    figure, axis = plt.subplots(figsize=(8, 4.5))
    axis.plot(positions, sweep.train, "o-", label="training accuracy")
    axis.plot(positions, sweep.test, "s-", label="test accuracy")
    axis.axhline((test_y == 0).mean(), color="gray", linestyle=":",
                 label="always guess 'died'")
    axis.set_xticks(list(positions))
    axis.set_xticklabels(labels)
    axis.set_xlabel("max_depth")
    axis.set_ylabel("accuracy")
    axis.set_title("One tree, ten depths")
    axis.legend()
    plt.show()
except NameError:
    print("sweep is not defined yet. Run the cell above first.")

### What the figure shows

Training accuracy keeps climbing, and with no depth limit it reaches 98.3%. Test accuracy peaks at depth 4 and then falls, ending below the accuracy of depth 1.
What the tree is learning from the training set with greater depth clearly does not **generalize** to the test set.

In other words, the deep tree is not learning more about surviving a shipwreck. It is learning about
the 712 particular people in the training set. With no limit it grows 145 leaves
for 712 passengers, which is roughly five passengers per leaf, and many leaves
hold a single person. A leaf describing just one person is unlikely to generalize to other passengers.

This phenomenon is **overfitting**.
Notice that training accuracy alone gives no warning at all as it monotonically increases.

## Explain

Answer each question in 3 to 6 sentences, in the markdown cell that follows it.

**1.** In Part 2, splitting the eight passengers on `Pclass` gained 0.1887 bits
while splitting on `Sex` gained 0.5488. In Part 4, the real tree also split on
sex first, but used `Pclass` lower down. Explain how a feature can be the weaker
choice at the root and still be a good choice later on.

*Your answer here.*

**2.** The unlimited-depth tree classified 98.3% of the training passengers
correctly and 72.6% of the test passengers. Suppose a classmate reports that the classifier works with accuracy 98%. What is wrong with this claim?

*Your answer here.*